# Lab 2: Multi-Class Human Activity Recognition & Model Quantization Scheme Analysis
---

## Section 1: Data Acquisition Overview
Implementing an on-device embedded AI pipeline to classify three activities:
1. **Idle / Sitting** (Class 0)
2. **Walking** (Class 1)
3. **Sit-ups** (Class 2)



## Sensor Ingestion, Magnitude Calculation & Feature Extraction
In this step:
1. Raw tri-axial accelerometer streams ($a_x, a_y, a_z$) are parsed from `idle.csv`, `walking.csv`, and `situps.csv`.
2. First and last 3 seconds are trimmed to eliminate phone handling artifacts.
3. Orientation independence is achieved by calculating the 3D acceleration vector magnitude:
   $$a_{\text{mag}} = \sqrt{a_x^2 + a_y^2 + a_z^2}$$
4. For every 1-second interval (100 samples at 100 Hz, with 50% stride), three statistical features are extracted:
   - **Mean ($\mu$):** $\mu = \frac{1}{N}\sum_{i=1}^{N} a_{\text{mag}, i}$
   - **Standard Deviation ($\sigma$):** $\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N} (a_{\text{mag}, i} - \mu)^2}$
   - **Root Mean Square (RMS):** $\text{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} a_{\text{mag}, i}^2}$

In [ ]:
# Data Loading & Feature Engineering
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def extract_features_from_csv(csv_path, label, window_size=100, step_size=50):
    df = pd.read_csv(csv_path)

    # Identify axis columns dynamically
    cols = {str(c).lower().strip(): c for c in df.columns}
    x_col = [cols[k] for k in cols if 'x' in k][0]
    y_col = [cols[k] for k in cols if 'y' in k][0]
    z_col = [cols[k] for k in cols if 'z' in k][0]

    # Trim first and last 3 seconds (300 samples)
    trim = 300
    if len(df) > (2 * trim + window_size):
        df = df.iloc[trim:-trim].reset_index(drop=True)

    ax = df[x_col].values.astype(np.float32)
    ay = df[y_col].values.astype(np.float32)
    az = df[z_col].values.astype(np.float32)

    # 3D Vector Magnitude
    a_mag = np.sqrt(ax**2 + ay**2 + az**2)

    features = []
    labels = []

    # Windowing
    for start in range(0, len(a_mag) - window_size + 1, step_size):
        w = a_mag[start : start + window_size]
        mu = np.mean(w)
        sigma = np.std(w)
        rms = np.sqrt(np.mean(w**2))

        features.append([mu, sigma, rms])
        labels.append(label)

    return np.array(features, dtype=np.float32), np.array(labels, dtype=np.int32)

# Extract from all 3 files
X_idle, y_idle = extract_features_from_csv('Idle_raw_data.csv', label=0)
X_walk, y_walk = extract_features_from_csv('Walk_Raw_Data.csv', label=1)
X_sit,  y_sit  = extract_features_from_csv('Situps_Raw_Data.csv',  label=2)

X = np.vstack([X_idle, X_walk, X_sit])
y = np.concatenate([y_idle, y_walk, y_sit])

print(f"Total window samples extracted: {X.shape[0]}")
print(f"Counts per class -> Idle: {len(y_idle)}, Walking: {len(y_walk)}, Sit-ups: {len(y_sit)}")

# Train/Test Split (80/20 Stratified) & Standardization
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

Total window samples extracted: 5131
Counts per class -> Idle: 1872, Walking: 2130, Sit-ups: 1129


## Baseline Multi-Layer Perceptron (MLP) Model
An ultra-lightweight Multi-Layer Perceptron is designed to fit embedded microcontroller memory constraints:
- **Input Layer:** 3 nodes (Mean, Standard Deviation, RMS)
- **Hidden Layer 1:** 16 units, ReLU activation
- **Hidden Layer 2:** 8 units, ReLU activation
- **Output Layer:** 3 units, Softmax activation (Idle, Walking, Sit-ups)



In [ ]:
# Train Baseline MLP Classifier
tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=60,
    batch_size=16,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)

loss, acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nFinal Baseline Model Test Accuracy: {acc * 100:.2f}%")

# Export for TFLite conversion
model.export('baseline_saved_model')

Epoch 1/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.7875 - loss: 0.5615 - val_accuracy: 0.8559 - val_loss: 0.3974
Epoch 2/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8572 - loss: 0.3724 - val_accuracy: 0.8627 - val_loss: 0.3718
Epoch 3/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8587 - loss: 0.3564 - val_accuracy: 0.8656 - val_loss: 0.3591
Epoch 4/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8621 - loss: 0.3464 - val_accuracy: 0.8724 - val_loss: 0.3522
Epoch 5/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8638 - loss: 0.3400 - val_accuracy: 0.8734 - val_loss: 0.3460
Epoch 6/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8662 - loss: 0.3359 - val_accuracy: 0.8773 - val_loss: 0.3414
Epoch 7/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8665 - loss: 0.3329 - val_accuracy: 0.8793 - val_loss: 0.3378
Epoch 8/60
257/257 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8677 - loss: 0.3309 - val_accuracy: 0

## TensorFlow Lite Quantization Schemes
Four variants are generated from the saved baseline model:
1. **Variant A (Baseline Float32):** Standard unquantized model using 32-bit IEEE floating-point numbers.
2. **Variant B (Dynamic Range Quantization):** Post-training weight quantization to 8-bit integers; activations remain in 32-bit float during inference.
3. **Variant C (Float16 Quantization):** Weights converted to 16-bit floating-point numbers, halving weight footprint with minimal precision loss.
4. **Variant D (Full Integer Quantization - PTQ):** Both weights and activations are quantized to 8-bit integers using a representative calibration dataset to profile dynamic activation ranges.

In [ ]:
#  Generate 4 TFLite Model Variants

# 1. Variant A: Baseline Float32
conv_a = tf.lite.TFLiteConverter.from_saved_model('baseline_saved_model')
tflite_a = conv_a.convert()
with open('model_float32.tflite', 'wb') as f:
    f.write(tflite_a)

# 2. Variant B: Dynamic Range Quantization
conv_b = tf.lite.TFLiteConverter.from_saved_model('baseline_saved_model')
conv_b.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_b = conv_b.convert()
with open('model_dynamic_range.tflite', 'wb') as f:
    f.write(tflite_b)

# 3. Variant C: Float16 Quantization
conv_c = tf.lite.TFLiteConverter.from_saved_model('baseline_saved_model')
conv_c.optimizations = [tf.lite.Optimize.DEFAULT]
conv_c.target_spec.supported_types = [tf.float16]
tflite_c = conv_c.convert()
with open('model_float16.tflite', 'wb') as f:
    f.write(tflite_c)

# 4. Variant D: Full Integer Quantization (int8)
def representative_data_gen():
    for row in X_train_scaled:
        yield [np.expand_dims(row, axis=0)]

conv_d = tf.lite.TFLiteConverter.from_saved_model('baseline_saved_model')
conv_d.optimizations = [tf.lite.Optimize.DEFAULT]
conv_d.representative_dataset = representative_data_gen
conv_d.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_d.inference_input_type = tf.int8
conv_d.inference_output_type = tf.int8
tflite_d = conv_d.convert()
with open('model_full_integer.tflite', 'wb') as f:
    f.write(tflite_d)

print("All 4 TFLite quantization models successfully compiled.")

All 4 TFLite quantization models successfully compiled.


## Comparative Benchmarking & Deliverables Table
Each compiled model is evaluated over the hold-out test set ($X_{\text{test}}$) to record:
- **Model Size (KB):** Total ROM/Flash storage footprint.
- **Accuracy (%):** Classification accuracy on the test partition.
- **Memory Reduction (%):** Compression ratio relative to the Float32 baseline.
- **Inference Latency ($\mu$s):** Mean single-window execution time measured using high-resolution hardware counters (`time.perf_counter`).

In [ ]:
# Automated Benchmarking Pipeline
def evaluate_tflite_model(model_path, X_eval, y_eval):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]

    correct = 0
    latencies = []
    is_int8 = (in_details['dtype'] == np.int8)

    for i in range(len(X_eval)):
        sample = np.expand_dims(X_eval[i], axis=0)

        # Scale input into int8 quantization space for Variant D
        if is_int8:
            scale, zero_point = in_details['quantization']
            sample = np.round(sample / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(in_details['index'], sample)

        # Measure inference execution time
        t0 = time.perf_counter()
        interpreter.invoke()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1e6)  # microseconds

        output = interpreter.get_tensor(out_details['index'])
        if np.argmax(output) == y_eval[i]:
            correct += 1

    acc = (correct / len(X_eval)) * 100.0
    latency = np.mean(latencies)
    size_kb = os.path.getsize(model_path) / 1024.0

    return size_kb, acc, latency

experiments = [
    ("Baseline (Unquantized)", "Float32", "model_float32.tflite"),
    ("Dynamic Range", "Weights int8, Act float32", "model_dynamic_range.tflite"),
    ("Float16 Quantization", "Float16", "model_float16.tflite"),
    ("Full Integer (PTQ)", "Weights int8, Act int8", "model_full_integer.tflite")
]

table_rows = []
base_size = None

for name, dtype_label, filepath in experiments:
    sz, accuracy, lat = evaluate_tflite_model(filepath, X_test_scaled, y_test)
    if base_size is None:
        base_size = sz
        mem_red = "0.0%"
    else:
        red_val = ((base_size - sz) / base_size) * 100.0
        mem_red = f"{red_val:.1f}%"

    table_rows.append({
        "Quantization Scheme": name,
        "Target Data Type": dtype_label,
        "Model Size (KB)": f"{sz:.2f}",
        "Accuracy (%)": f"{accuracy:.2f}%",
        "Memory Reduction (%)": mem_red,
        "Avg Latency (µs)": f"{lat:.1f}"
    })

benchmarking_table = pd.DataFrame(table_rows)
print("\n" + "="*80)
print("BENCHMARKING RESULTS TABLE (DELIVERABLE 4)")
print("="*80)
print(benchmarking_table.to_markdown(index=False))


BENCHMARKING RESULTS TABLE (DELIVERABLE 4)
| Quantization Scheme    | Target Data Type          |   Model Size (KB) | Accuracy (%)   | Memory Reduction (%)   |   Avg Latency (µs) |
|:-----------------------|:--------------------------|------------------:|:---------------|:-----------------------|-------------------:|
| Baseline (Unquantized) | Float32                   |              2.94 | 87.63%         | 0.0%                   |                5.9 |
| Dynamic Range          | Weights int8, Act float32 |              2.94 | 87.63%         | 0.0%                   |                1.6 |
| Float16 Quantization   | Float16                   |              3.25 | 87.63%         | -10.8%                 |                1.6 |
| Full Integer (PTQ)     | Weights int8, Act int8    |              3.27 | 86.76%         | -11.2%                 |                2   |


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Technical Report & Analysis

### Why Full Integer Quantization (PTQ) is Mandatory for Microcontroller Deployment

1. **Absence of Hardware Floating Point Units (FPUs):**
   Low-power microcontrollers (e.g., ARM Cortex-M0/M0+, Cortex-M3, low-power ESP32 variants) do not possess a dedicated hardware FPU. Executing floating-point operations requires software emulation routines that consume dozens of clock cycles per basic operation. Full Integer Quantization uses purely integer arithmetic (e.g., SIMD instructions via CMSIS-NN), reducing cycle count and instruction latency.

2. **Severe Memory (SRAM & Flash) Constraints:**
   Typical microcontrollers operate with $32\text{ KB} - 64\text{ KB}$ of SRAM and $256\text{ KB} - 512\text{ KB}$ of Flash storage. Dynamic Range Quantization and Float16 retain floating-point buffers in SRAM for runtime activations. Full integer quantization shrinks both weights and intermediate activation tensors to 8 bits, minimizing total peak SRAM allocation during model execution.

3. **Energy & Battery Longevity:**
   Integer ALU operations consume significantly fewer picojoules ($pJ$) of energy per Multiply-Accumulate (MAC) operation compared to 32-bit floating-point math. For a wearable device powered by a coin-cell or small LiPo battery, integer execution directly reduces thermal load and maximizes operational battery life.